# Smartphone Addiction - Tuned CatBoost

A single CatBoost model: IterativeImputer columns fitted inside each fold, hyperparameters
tuned with Optuna, then a full 5-fold CV with 2-seed bagging.

Answers a question nothing else in this repo has: the iterative-imputed columns were worth
+0.00065 AUC **on XGBoost** (`results.md`), and CatBoost is still the LB-best single model
(0.96279 CV from bare defaults) but has never been validly tuned. This combines both.

No stacking, so the OOF AUC is directly comparable to the 0.96279 baseline row.
Setup guide: `KAGGLE_SETUP.md`.

In [ ]:
import gc, glob, os, subprocess, time, warnings
import numpy as np
import pandas as pd
import optuna
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
SEEDS = [42, 2024]
N_SPLITS = 5
ESR = 300
IMP_ITER = 20
ITERS = 30000

MAX_HOURS = 8.0
TUNE_HOURS = 3.0
STUDY_FRAC = 0.70          # rest of TUNE_HOURS is reserved for the re-eval gate
TUNE_LR = 0.10             # screening: ~3x fewer iterations to converge
FINAL_LR = 0.03            # deployment
TUNE_SAMPLE = 200_000      # rows A - the search
SEL_SAMPLE = 200_000       # rows B - the gate, disjoint from A
TUNE_FOLDS = 3
TUNE_PATIENCE = 15

TARGET = "addicted_label"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
NUM_COLS = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
            "work_study_hours", "sleep_hours", "notifications_per_day",
            "app_opens_per_day", "weekend_screen_time"]

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
np.random.seed(SEED)

T0 = time.time()
def elapsed_s():   return time.time() - T0
def remaining_s(): return MAX_HOURS * 3600 - elapsed_s()

print(f"budget {MAX_HOURS}h (tune {TUNE_HOURS}h)  screen lr={TUNE_LR}  final lr={FINAL_LR}")

In [ ]:
def has_cuda():
    try:
        subprocess.check_output(["nvidia-smi"], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

CUDA = has_cuda()
N_JOBS = os.cpu_count()
print(f"cuda={CUDA}  cores={N_JOBS}  task_type={'GPU' if CUDA else 'CPU'}")

In [ ]:
def find_data():
    known = ["/kaggle/input/competitions/playground-series-s6e8",
             "/kaggle/input/playground-series-s6e8", "../data", "data"]
    cands = [f"{d}/train.csv" for d in known]
    cands += sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))
    for f in cands:
        d = os.path.dirname(f)
        if os.path.exists(f) and os.path.exists(f"{d}/test.csv"):
            return d
    seen = sorted(glob.glob("/kaggle/input/**", recursive=True))[:40]
    raise FileNotFoundError("train.csv + test.csv not found. /kaggle/input holds:\n" + "\n".join(seen))

DATA = find_data()
train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")

y = train.pop(TARGET).values
test_ids = test["id"].values
X = train.drop(columns="id")
X_test = test.drop(columns="id")[X.columns]

for d in (X, X_test):
    for c in NUM_COLS:
        d[c] = d[c].astype("float32")

for c in CAT_COLS:
    levels = sorted(set(X[c].dropna()) | set(X_test[c].dropna()))
    X[c] = pd.Categorical(X[c], categories=levels)
    X_test[c] = pd.Categorical(X_test[c], categories=levels)

print(f"train {X.shape}  test {X_test.shape}  base rate {y.mean():.4f}  nan {X.isna().sum().sum():,}")

In [ ]:
IMP_COLS = [c + "_imp" for c in NUM_COLS]

def add_imputed(X_fit, frames, verbose=True):
    # Fitted on the training fold only. Raw NaN columns are KEPT alongside the imputed
    # ones: CatBoost learns a split direction for NaN, so the model sees both
    # "is this missing" and "what would it have been".
    imp = IterativeImputer(max_iter=IMP_ITER, random_state=SEED).fit(X_fit[NUM_COLS])
    if verbose:
        print(f"  imputer n_iter={imp.n_iter_}/{IMP_ITER}"
              + ("  <-- did not converge" if imp.n_iter_ >= IMP_ITER else ""))
    out = []
    for d in frames:
        d = d.copy()
        d[IMP_COLS] = imp.transform(d[NUM_COLS]).astype("float32")
        out.append(d)
    return out

def to_catboost(d):
    d = d.copy()
    for c in CAT_COLS:
        d[c] = d[c].astype(object).fillna("missing").astype(str)
    return d

In [ ]:
DEFAULTS = dict(depth=8, l2_leaf_reg=6.0, random_strength=1.0,
                border_count=128, one_hot_max_size=2,
                bootstrap_type="Bayesian", bagging_temperature=1.0)

def merge(p):
    # The ONLY place DEFAULTS is applied. bagging_temperature is valid only under the
    # Bayesian bootstrap (CatBoost raises otherwise) and subsample only under Bernoulli,
    # so the illegal partner is always dropped here. Idempotent.
    q = {**DEFAULTS, **(p or {})}
    if q.get("bootstrap_type") == "Bayesian":
        q.pop("subsample", None)
    else:
        q.pop("bagging_temperature", None)
    return q

def make_cat(seed, p=None, lr=FINAL_LR, iters=ITERS):
    return cb.CatBoostClassifier(
        iterations=iters, learning_rate=lr, eval_metric="AUC", random_seed=seed,
        verbose=False, allow_writing_files=False,
        task_type="GPU" if CUDA else "CPU",
        **merge(p))

def fit_cat(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=CAT_COLS,
          early_stopping_rounds=ESR, use_best_model=True, verbose=False)
    return m

def space(t):
    # bootstrap_type gates its own sampling parameter: bagging_temperature is only read
    # under Bayesian (silently ignored otherwise), and passing it with Bernoulli raises.
    p = dict(
        depth            = t.suggest_int("depth", 4, 8),
        l2_leaf_reg      = t.suggest_float("l2_leaf_reg", 1, 20, log=True),
        random_strength  = t.suggest_float("random_strength", 0, 5),
        border_count     = t.suggest_int("border_count", 128, 254),
        one_hot_max_size = t.suggest_int("one_hot_max_size", 2, 10),
        bootstrap_type   = t.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli"]),
    )
    if p["bootstrap_type"] == "Bayesian":
        p["bagging_temperature"] = t.suggest_float("bagging_temperature", 0, 2)
    else:
        p["subsample"] = t.suggest_float("subsample", 0.5, 1.0)
    return p


In [ ]:
def build_folds(rows, n_splits):
    Xr = X.iloc[rows].reset_index(drop=True)
    yr = y[rows]
    out = []
    for tr, va in StratifiedKFold(n_splits, shuffle=True, random_state=SEED).split(Xr, yr):
        A, B = add_imputed(Xr.iloc[tr], [Xr.iloc[tr], Xr.iloc[va]], verbose=False)
        out.append((to_catboost(A), to_catboost(B), yr[tr], yr[va]))
    return out

def score_params(p, folds, lr, trial=None):
    aucs = []
    for i, (A, B, ytr, yva) in enumerate(folds):
        m = fit_cat(make_cat(SEED, p, lr=lr), A, ytr, B, yva)
        aucs.append(roc_auc_score(yva, m.predict_proba(B)[:, 1]))
        del m; gc.collect()
        if trial is not None:
            trial.report(float(np.mean(aucs)), step=i)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return float(np.mean(aucs))

def make_stopper(patience):
    st = {"best": -np.inf, "since": 0}
    def cb_(study, trial):
        try:
            v = study.best_value
        except ValueError:
            return
        if v > st["best"] + 1e-6:
            st["best"], st["since"] = v, 0
        else:
            st["since"] += 1
            if st["since"] >= patience:
                study.stop()
    return cb_

FINAL_PARAMS = dict(DEFAULTS)

if TUNE_HOURS > 0:
    perm = np.random.RandomState(SEED).permutation(len(X))
    rows_A = perm[:TUNE_SAMPLE]
    rows_B = perm[TUNE_SAMPLE:TUNE_SAMPLE + SEL_SAMPLE]
    print(f"building search folds (A={len(rows_A):,}) and gate folds (B={len(rows_B):,})")
    FOLDS_A = build_folds(rows_A, TUNE_FOLDS)
    FOLDS_B = build_folds(rows_B, TUNE_FOLDS)

    t_m = time.time()
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=0))
    study.optimize(lambda t: score_params(space(t), FOLDS_A, TUNE_LR, trial=t),
                   timeout=min(TUNE_HOURS * 3600 * STUDY_FRAC, remaining_s() * 0.5),
                   callbacks=[make_stopper(TUNE_PATIENCE)],
                   catch=(Exception,))

    st = [t.state.name for t in study.trials]
    print(f"\nstudy: {len(study.trials)} trials in {(time.time()-t_m)/60:.1f} min  "
          f"({st.count('COMPLETE')} complete, {st.count('PRUNED')} pruned, {st.count('FAIL')} failed)")
    if st.count("FAIL") > 0.15 * max(len(st), 1):
        print(f"  WARNING: {st.count('FAIL')}/{len(st)} trials FAILED - catch=(Exception,) hides the "
              f"reason. Re-run one config manually to see it; a systematically invalid parameter "
              f"combination silently burns the whole tuning budget.")

    # Pruned trials DO carry a .value - an intermediate single-fold score, not a 3-fold
    # mean - so filter on state, never on "value is not None".
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if not done:
        print("no trial completed - keeping defaults")
    else:
        # Best-of-k inflates the winner by ~sigma*sqrt(2 ln k). Re-score the top 3 AND the
        # defaults on disjoint rows B, at the FINAL lr - so the gate also tests whether the
        # params survive the learning-rate change they were not screened at.
        top = sorted(done, key=lambda t: t.value, reverse=True)[:3]
        cands = [("defaults", dict(DEFAULTS))]
        cands += [(f"trial{t.number}", dict(t.params)) for t in top]
        scored = []
        for lbl, p in cands:
            if remaining_s() < 0.15 * MAX_HOURS * 3600:
                print(f"   gate: out of budget, skipping {lbl}")
                continue
            scored.append((score_params(p, FOLDS_B, FINAL_LR), lbl, p))
        scored.sort(key=lambda z: z[0], reverse=True)

        print(f"study best (on A, lr={TUNE_LR}): {study.best_value:.5f}")
        for auc, lbl, _ in scored:
            print(f"   gate on B (lr={FINAL_LR})  {lbl:10s} {auc:.5f}")
        if scored:
            FINAL_PARAMS = merge(scored[0][2])
            print(f"   -> adopted: {scored[0][1]}")

    del FOLDS_A, FOLDS_B
    gc.collect()

print(f"\nfinal params: {FINAL_PARAMS}")
print(f"elapsed {elapsed_s()/60:.1f} min")

In [ ]:
oof = np.zeros(len(X))
mask = np.zeros(len(X), dtype=bool)
test_sum = np.zeros(len(X_test))
fold_scores, all_iters, done_folds = [], [], []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
t_loop = time.time()
ran = 0

for fold, (tr, va) in enumerate(skf.split(X, y)):
    if ran:
        avg = (time.time() - t_loop) / ran
        if elapsed_s() + avg > MAX_HOURS * 3600:
            print(f"next fold would exceed {MAX_HOURS}h - stopping with {len(done_folds)} folds")
            break

    t0 = time.time()
    ytr, yva = y[tr], y[va]
    Xtr, Xva, Xte = add_imputed(X.iloc[tr], [X.iloc[tr], X.iloc[va], X_test])
    A, B, C = (to_catboost(d) for d in (Xtr, Xva, Xte))

    pv, pt, bis = np.zeros(len(va)), np.zeros(len(X_test)), []
    for s in SEEDS:
        m = fit_cat(make_cat(s, FINAL_PARAMS, lr=FINAL_LR), A, ytr, B, yva)
        pv += m.predict_proba(B)[:, 1] / len(SEEDS)
        pt += m.predict_proba(C)[:, 1] / len(SEEDS)
        bis.append(m.get_best_iteration())
        del m; gc.collect()

    oof[va] = pv
    mask[va] = True
    test_sum += pt
    fold_scores.append(roc_auc_score(yva, pv))
    all_iters += bis
    done_folds.append(fold)
    ran += 1

    flag = f"  <-- HIT CAP {ITERS}, NOT CONVERGED" if max(bis) >= ITERS - 1 else ""
    print(f"fold {fold+1}/{N_SPLITS}  AUC={fold_scores[-1]:.5f}  best_iter={bis}  "
          f"({time.time()-t0:.0f}s){flag}")
    del Xtr, Xva, Xte, A, B, C
    gc.collect()

print(f"\ntotal {elapsed_s()/60:.1f} min, {len(done_folds)} folds")

In [ ]:
assert done_folds, "no folds completed"

oof_auc = roc_auc_score(y[mask], oof[mask])
print(f"OOF AUC      {oof_auc:.5f}   <-- compare against the 0.96279 CatBoost baseline")
print(f"fold AUCs    {np.round(fold_scores, 5)}")
print(f"mean +/- std {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")
print(f"best_iters   {all_iters}")
if max(all_iters) >= ITERS - 1:
    print(f"WARNING: hit the {ITERS}-iteration cap - this AUC is a floor, raise ITERS")
print(f"params       {FINAL_PARAMS}")

In [ ]:
pred = test_sum / len(done_folds)

sub = pd.DataFrame({"id": test_ids, TARGET: pred})
assert len(sub) == len(X_test), f"expected {len(X_test)} rows, got {len(sub)}"
assert sub["id"].is_unique
assert sub[TARGET].notna().all()
assert sub[TARGET].between(0, 1).all()
assert sub[TARGET].nunique() > 2

sub.to_csv(f"{WORK}/submission.csv", index=False)
print(f"{len(sub):,} rows  mean {pred.mean():.4f}  (train base rate {y.mean():.4f})")
if abs(pred.mean() - y.mean()) > 0.03:
    print("WARNING: mean prediction far from base rate - inspect before submitting")
print(f"folds used: {len(done_folds)}/{N_SPLITS}")
sub.head()